# Supply Risk Control Tower | Colab lab
Run cells in order. The default path runs on CPU with deterministic synthetic data and cited extractive answers. The optional Qwen cell downloads model weights and enables generation. Select a GPU runtime for faster inference.

You will see the dataset catalog and joins, a chronological regression backtest, forecast intervals, a ranked exception queue, evidence retrieval and reranking, a planner dashboard and traces. All simulated dates, policies, costs and outcomes are illustrative. A public shipment CSV can be profiled separately using the optional import cell.

In [ ]:
# Run in a fresh Colab runtime if dependencies are missing.
%pip -q install xgboost scikit-learn matplotlib pandas ipywidgets transformers accelerate

## 1. Data economy
Daily demand, inventory and inbound receipts share the grain **date × plant × SKU**. Plant, SKU and supplier are dimensions. Finance is illustrative allocated revenue and cost per row. Contracts and shipments use unique business IDs and are evaluated as a separate snapshot to avoid duplicating their alerts across dates. An approved playbook is the retrieval corpus. The seed and validation checks make the simulation reproducible.

In [ ]:
import json, time, re, hashlib, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

SEED = 42
rng = np.random.default_rng(SEED)
plants = pd.DataFrame({'plant':['Bengaluru','Houston','Rotterdam','Singapore'],
                       'region':['Asia','Americas','Europe','Asia']})
suppliers = pd.DataFrame({'supplier':[f'V{i:02}' for i in range(8)],
                          'vendor_score':[86,95,91,87,92,99,89,96]})
materials = pd.DataFrame({'sku':[f'SKU-{100+i}' for i in range(24)],
                          'supplier':np.repeat(suppliers.supplier,3),
                          'unit_price':rng.integers(45,180,24),
                          'criticality':rng.choice(['critical','standard'],24)})
dates = pd.date_range('2025-01-01', periods=240)
facts = pd.MultiIndex.from_product([dates,plants.plant,materials.sku],names=['date','plant','sku']).to_frame(index=False)
facts = facts.merge(plants,on='plant',validate='many_to_one').merge(materials,on='sku',validate='many_to_one').merge(suppliers,on='supplier',validate='many_to_one')
day = (facts.date-dates[0]).dt.days
sku_idx = facts.sku.str[-2:].astype(int)
base = 40 + sku_idx*3 + facts.plant.map(dict(zip(plants.plant,[16,8,0,12])))
signal = base + 8*np.sin(2*np.pi*day/7) + .09*day
facts['demand'] = np.maximum(0,np.rint(signal+rng.normal(0,8,len(facts)))).astype(int)
facts['stock'] = np.maximum(0,np.rint(signal+rng.normal(-14,22,len(facts)))).astype(int)
facts['inbound'] = rng.integers(0,35,len(facts))
facts['fulfilled'] = np.minimum(facts.demand,facts.stock+facts.inbound)
facts['revenue'] = facts.fulfilled*facts.unit_price
facts['cogs'] = facts.revenue*.58
facts['opex'] = facts.revenue*.16
facts['tax'] = (facts.revenue-facts.cogs-facts.opex)*.25
facts['pat'] = facts.revenue-facts.cogs-facts.opex-facts.tax
facts['shortage'] = (facts.demand-facts.stock-facts.inbound).clip(lower=0)
contracts = suppliers[['supplier']].assign(contract_id=[f'C{i:02}' for i in range(8)],days_to_expiry=[-2,18,42,70,130,220,59,119])
shipments = facts[facts.date.eq(dates[-1])][['plant','sku','supplier']].reset_index(drop=True)
shipments['shipment_id'] = [f'S{i:03}' for i in shipments.index]
shipments['downtime_days'] = rng.integers(0,27,len(shipments))
shipments['cause'] = rng.choice(['Customs hold','Carrier capacity','Warehouse backlog','Weather'],len(shipments))
shipments['otif'] = shipments.downtime_days.le(7) & (rng.random(len(shipments))>.07)
knowledge = [{'id': 'K01', 'title': 'OTIF breach', 'tags': ['otif', 'late', 'delivery', 'performance'], 'text': 'Confirm promised date and in-full quantity against receipt events. Split late and short lines by carrier, vendor and plant. For critical demand, confirm alternate shipment capacity and obtain planner approval before promising a revised date.'}, {'id': 'K02', 'title': 'Vendor scorecard escalation', 'tags': ['vendor', 'supplier', 'scorecard', 'quality'], 'text': 'Review the quality, delivery and compliance components and compare against the contracted service level. Request a corrective action plan with an owner and due date. Qualify alternate suppliers before switching allocation.'}, {'id': 'K03', 'title': 'Logistics downtime and cause triage', 'tags': ['logistics', 'downtime', 'carrier', 'customs', 'weather', 'warehouse', 'equipment'], 'text': 'Confirm last physical scan and ETA with the carrier. For customs, verify the missing document with the broker. For equipment failure or capacity shortage, check a replacement vehicle or alternate carrier. For weather, validate a safe route before rerouting. At critical downtime, escalate to the regional logistics owner and check which plants will run out first.'}, {'id': 'K04', 'title': 'Contract expiry and continuity', 'tags': ['contract', 'expiry', 'expired', 'renewal'], 'text': 'At 120 days validate coverage and sourcing lead time. At 60 days confirm renewal or tender ownership. At 30 days escalate to the category lead and agree temporary coverage if authorized. Review expired contracts with the contract owner before further orders.'}, {'id': 'K05', 'title': 'Stock-out and demand scheduling', 'tags': ['stock', 'stockout', 'shortfall', 'demand', 'inventory', 'gap', 'supply', 'bottleneck'], 'text': 'Validate usable stock, confirmed inbound ETA, and demand due date. Sequence constrained demand by service commitment, criticality and available stock. Check interplant transfer availability, approved substitutions and expedite cost. The plant planner signs off on reallocations and updates the schedule after capacity is confirmed.'}, {'id': 'K06', 'title': 'Turnover and slow inventory', 'tags': ['turnover', 'slow', 'turns', 'ratio', 'obsolescence'], 'text': 'Check the reporting period and average inventory denominator for turnover. Review aged inventory value by plant, material and expiry date. Test reuse at another plant and pause replenishment for verified excess. Finance approves any write-down.'}]
assert not facts.duplicated(['date','plant','sku']).any()
assert facts[['demand','stock','inbound']].ge(0).all().all()
assert facts.supplier.isin(suppliers.supplier).all()
assert not facts.isna().any().any()
catalog = pd.DataFrame([(name,len(table),', '.join(table.columns)) for name,table in
    [('plant',plants),('material',materials),('supplier',suppliers),('daily fact',facts),('contract snapshot',contracts),('shipment snapshot',shipments),('playbooks',pd.DataFrame(knowledge))]],columns=['dataset','rows','columns'])
display(catalog, facts.tail(4))

### Optional public shipment profile
Set a CSV path (uploaded into Colab) or an HTTPS CSV URL. The profile is separate from the synthetic forecasting data. Delivery dates can establish lateness, while in-full delivery requires quantity evidence.

In [ ]:
PUBLIC_CSV = ''  # Example: /content/SCMS_Delivery_History_Dataset.csv
if PUBLIC_CSV:
    public = pd.read_csv(PUBLIC_CSV, low_memory=False)
    scheduled = pd.to_datetime(public['Scheduled Delivery Date'],errors='coerce',format='mixed')
    delivered = pd.to_datetime(public['Delivered to Client Date'],errors='coerce',format='mixed')
    valid = scheduled.notna() & delivered.notna()
    display(pd.DataFrame({'rows':[len(public)],'comparable_dates':[valid.sum()],
                          'late_rate':[float((delivered[valid]>scheduled[valid]).mean())]}))
    if 'Country' in public: display(public.Country.value_counts().head(12))
else:
    print('Public CSV import skipped. Demo data is synthetic.')

## 2. Forecast and evaluate
One-day-ahead demand forecasts use lagged actuals, weekday and plant/SKU identifiers. Train, validation, interval calibration and test periods are chronological. Earlier actuals are available at each test prediction time, so this measures rolling one-step performance. The naive baseline uses demand seven days earlier. Selection uses validation MAE. The interval uses absolute residuals from a separate calibration window and reports observed test coverage. Future stock and inbound are planning assumptions carried forward from the latest snapshot.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor
import joblib
try:
    from xgboost import XGBRegressor
    boosted = XGBRegressor(n_estimators=110,max_depth=5,learning_rate=.08,random_state=SEED,n_jobs=2)
    boost_name = 'XGBoost'
except ImportError:
    boosted = HistGradientBoostingRegressor(max_iter=110,random_state=SEED)
    boost_name = 'Histogram gradient boosting (fallback)'

groups = facts.groupby(['plant','sku'],sort=False).demand
for lag in (1,7,14): facts[f'lag{lag}'] = groups.shift(lag)
facts['weekday'] = facts.date.dt.dayofweek
facts['trend'] = (facts.date-dates[0]).dt.days
cats = ['plant','sku','weekday']
nums = ['trend','lag1','lag7','lag14']
features = cats+nums
frame = facts.dropna(subset=features)
train = frame[frame.date<dates[160]]
val = frame[frame.date.between(dates[160],dates[184])]
cal = frame[frame.date.between(dates[185],dates[209])]
test = frame[frame.date>=dates[210]].copy()
assert train.date.max()<val.date.min()<cal.date.min()<test.date.min()
models = {'Ridge linear':Ridge(alpha=2),boost_name:boosted}
pipelines = {}
def predict(name, rows):
    return rows.lag7.to_numpy() if name=='Seasonal naive' else np.maximum(0,pipelines[name].predict(rows[features]))
for name, estimator in models.items():
    transform = ColumnTransformer([('categorical',OneHotEncoder(handle_unknown='ignore',sparse_output=False),cats)],remainder='passthrough')
    pipelines[name]=make_pipeline(transform,estimator).fit(train[features],train.demand)
comparison = pd.DataFrame([
    {'model':name,'validation_mae':mean_absolute_error(val.demand,predict(name,val)),
     'test_mae':mean_absolute_error(test.demand,predict(name,test)),
     'test_rmse':np.sqrt(mean_squared_error(test.demand,predict(name,test)))}
    for name in ['Seasonal naive',*models]]).sort_values('validation_mae')
champion = comparison.iloc[0].model
residual = np.abs(cal.demand-predict(champion,cal))
level = min(1,np.ceil((len(residual)+1)*.90)/len(residual))
radius = float(np.quantile(residual,level,method='higher'))
test['forecast'] = predict(champion,test)
test['lower'] = (test.forecast-radius).clip(lower=0)
test['upper'] = test.forecast+radius
coverage = float(test.demand.between(test.lower,test.upper).mean())
baseline_mae = comparison.set_index('model').loc['Seasonal naive','test_mae']
champion_mae = comparison.iloc[0].test_mae
display(comparison.round(3))
print(f'Champion: {champion} | nominal 90% interval ±{radius:.1f} units | observed test coverage {coverage:.1%}')
print('Promotion check:', 'PASS' if champion_mae<=baseline_mae and coverage>=.85 else 'REVIEW')


## 3. Build tomorrow’s risk queue
Point and upper-band demand gaps inform demand scheduling. Current inventory and inbound are carried forward for this scenario. Rules create stock-out, OTIF, vendor, downtime and contract alerts independently. Cross-plant stock is a candidate requiring reservation and transit checks.

In [ ]:
future = facts[facts.date.eq(dates[-1])].copy()
for lag in (1,7,14):
    lag_values = facts.loc[facts.date.eq(dates[-1]+pd.Timedelta(days=1-lag)),['plant','sku','demand']]
    future = future.drop(columns=f'lag{lag}').merge(lag_values.rename(columns={'demand':f'lag{lag}'}),on=['plant','sku'],validate='one_to_one')
future['date'] = dates[-1]+pd.Timedelta(days=1)
future['trend'] = len(dates)
future['weekday'] = future.date.dt.dayofweek
future['forecast'] = predict(champion,future)
future['upper'] = future.forecast+radius
future['available'] = future.stock+future.inbound
future['gap'] = (future.forecast-future.available).clip(lower=0)
future['upper_gap'] = (future.upper-future.available).clip(lower=0)
future['surplus'] = (future.available-future.upper).clip(lower=0)
alerts = []
def emit(plant,subject,kind,severity,detail,source):
    alerts.append(dict(plant=plant,subject=subject,kind=kind,severity=severity,detail=detail,source=source))
for r in future.itertuples():
    if r.upper_gap>0:
        donors = future[(future.sku==r.sku)&(future.plant!=r.plant)&(future.surplus>0)].sort_values('surplus',ascending=False)
        candidate = f' Candidate donor: {donors.iloc[0].plant}, {donors.iloc[0].surplus:.0f} units above its upper band.' if len(donors) else ''
        emit(r.plant,r.sku,'Stock-out' if r.stock==0 else 'Supply gap','Critical' if r.gap>0 and r.criticality=='critical' else 'Watch',
             f'Available {r.available}, point demand {r.forecast:.1f}, upper demand {r.upper:.1f}, point gap {r.gap:.1f}, upper gap {r.upper_gap:.1f}.'+candidate,'K05')
for plant, rows in shipments.groupby('plant'):
    otif=rows.otif.mean()*100
    if otif<95: emit(plant,'Shipment snapshot','OTIF','Watch',f'{otif:.1f}% on time and in full versus 95% target.','K01')
for r in suppliers.itertuples():
    if r.vendor_score<89: emit('Network',r.supplier,'Vendor score','Watch',f'{r.vendor_score}% versus 89% threshold.','K02')
for r in shipments.itertuples():
    if r.downtime_days>7:
        band=21 if r.downtime_days>21 else 14 if r.downtime_days>14 else 7
        emit(r.plant,r.shipment_id,'Downtime','Critical' if band==21 else 'Watch',f'{r.downtime_days} days, >{band} day tier. Cause: {r.cause}.','K03')
for r in contracts.itertuples():
    if r.days_to_expiry<=120:
        band=30 if r.days_to_expiry<=30 else 60 if r.days_to_expiry<=60 else 120
        emit('Network',r.contract_id,'Contract expiry','Critical' if band==30 else 'Watch',f'{r.days_to_expiry} days remaining, ≤{band} day tier.','K04')
queue = pd.DataFrame(alerts).sort_values(['severity','plant']).reset_index(drop=True)
queue.insert(0,'alert_id',[f'A{i:03}' for i in range(len(queue))])
display(queue.head(10))

## 4. Retrieval, reranking and orchestration
TF-IDF finds candidate passages. A second-stage scorer combines cosine similarity, query/tag overlap and a structured alert-policy match. Confidence is an **evidence score**, not a calibrated probability: 55% cosine + 25% tag coverage + 20% known policy match. Unrouted questions below 0.20 abstain. An optional Qwen generator receives only selected evidence and case facts. The citation gate checks retrieved IDs and falls back to evidence excerpts on failure. Citation presence does not verify entailment.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
tokenize = lambda text: set(re.findall(r'[a-z0-9]+',text.lower()))
trace = []
generator = None
def build_index(documents):
    global knowledge, vectorizer, corpus
    if not documents or len({d['id'] for d in documents})!=len(documents):
        raise ValueError('Provide a nonempty KB with unique IDs')
    for d in documents:
        if not re.fullmatch(r'[A-Za-z0-9_-]+',d['id']) or not d['text'] or not isinstance(d['tags'],list):
            raise ValueError('Invalid KB passage')
    v = TfidfVectorizer(ngram_range=(1,2),stop_words='english')
    matrix = v.fit_transform([d['title']+' '+d['text']+' '+' '.join(d['tags']) for d in documents])
    knowledge,vectorizer,corpus = documents,v,matrix
build_index(knowledge)
# Optional custom knowledge. JSON schema: id, title, text, tags.
KB_PATH = ''
if KB_PATH: build_index(json.loads(Path(KB_PATH).read_text()))
def answer(question, alert_id=None):
    start=time.perf_counter()
    row = queue.set_index('alert_id').loc[alert_id] if alert_id else None
    preferred = row.source if row is not None else None
    context = f'{row.plant} / {row.subject}. {row.detail}' if row is not None else ''
    query = question+(' '+row.kind if row is not None else '')
    sims=cosine_similarity(vectorizer.transform([query]),corpus)[0]
    candidate_indices = sorted(range(len(knowledge)),key=lambda i:sims[i],reverse=True)[:5]
    for i,d in enumerate(knowledge):
        if d['id']==preferred and i not in candidate_indices: candidate_indices.append(i)
    ranked=[]
    for i in candidate_indices:
        d=knowledge[i]
        tags=set(' '.join(d['tags']).lower().split())
        overlap=len(tokenize(query)&tags)/max(1,min(len(tags),3))
        match=d['id']==preferred
        score=.55*float(sims[i])+.25*overlap+.20*match
        ranked.append((float(score),d,float(sims[i]),overlap,match))
    ranked.sort(key=lambda x:x[0],reverse=True)
    hits=[d for score,d,*_ in ranked if score>=.20][:3]
    confidence=ranked[0][0]
    status='abstained'
    result='No sufficiently relevant approved passage. Escalate to the planner or add an approved policy.'
    generation_ms=0.0
    if hits:
        status='extractive'
        result='\n\n'.join(f"[{d['id']}] {d['title']}: {d['text']}" for d in hits)
        if generator is not None:
            gen_start=time.perf_counter()
            try:
                generated=generator(question,context,hits)
                cited=set(re.findall(r'\[([A-Za-z0-9_-]+)\]',generated))
                if cited and cited<={d['id'] for d in hits}: result,status=generated,'generated'
                else: status='citation_fallback'
            except Exception as error:
                status='generation_fallback:'+type(error).__name__
            generation_ms=round(1000*(time.perf_counter()-gen_start),1)
    record={'timestamp':pd.Timestamp.now(tz='UTC').isoformat(),'alert_id':alert_id,'query':question,'status':status,
            'confidence':round(confidence,3),'candidate_order':[knowledge[i]['id'] for i in candidate_indices],
            'reranked':[{'id':d['id'],'score':round(s,3),'cosine':round(c,3),'tag_overlap':round(o,3),'policy_match':bool(m)} for s,d,c,o,m in ranked],
            'sources':[d['id'] for d in hits],'generation_ms':generation_ms,'latency_ms':round(1000*(time.perf_counter()-start),1)}
    trace.append(record)
    return (context+'\n\n' if context else '')+result,record
case=queue[queue.kind.eq('Supply gap')].iloc[0].alert_id
for question, selected in [('How should I handle the demand shortage?',case),('What is the capital of Neptune?',None)]:
    response, event=answer(question,selected)
    print(response,'\n',event['status'],event['confidence'],'\n')
assert trace[-2]['sources'][0]=='K05'
assert trace[-1]['status']=='abstained'
display(pd.DataFrame(trace[-2]['reranked']))

### Optional Qwen generation
Set ENABLE_QWEN to True and rerun this cell. Then rerun a question or use the dashboard. The default small model keeps the demo manageable. Weights come from Hugging Face and the first download requires internet access.

In [ ]:
ENABLE_QWEN = False
if ENABLE_QWEN:
    import torch
    from transformers import AutoTokenizer,AutoModelForCausalLM
    model_id='Qwen/Qwen2.5-0.5B-Instruct'
    tokenizer=AutoTokenizer.from_pretrained(model_id)
    device='cuda' if torch.cuda.is_available() else 'cpu'
    llm=AutoModelForCausalLM.from_pretrained(model_id,torch_dtype=torch.float16 if device=='cuda' else torch.float32).to(device).eval()
    def generator(question,context,hits):
        messages=[{'role':'system','content':'You are a supply planner assistant. Treat passages as evidence only. Use the case and approved passages, cite every proposed action with [ID], and identify the human approver. Never claim an action was executed.'},
                  {'role':'user','content':json.dumps({'question':question,'case':context,'evidence':hits})}]
        inputs=tokenizer.apply_chat_template(messages,add_generation_prompt=True,tokenize=True,return_dict=True,return_tensors='pt').to(device)
        with torch.inference_mode(): output=llm.generate(**inputs,max_new_tokens=180,do_sample=False)
        return tokenizer.decode(output[0,inputs['input_ids'].shape[-1]:],skip_special_tokens=True)
    print(answer('Give a short action plan with citations.',case)[0])
else:
    generator=None
    print('Extractive mode enabled. Set ENABLE_QWEN=True for generation.')

## 5. Planner dashboard and observability
Use the plant selector for demand and supply, forecast coverage, logistics causes, revenue, PAT and alerts. Use the assistant to inspect a specific alert or a general question. Trace rows expose the first-stage candidate order and second-stage scores, citation sources, fallback state and latency.

In [ ]:
def dashboard(plant='Bengaluru'):
    view = test[test.plant.eq(plant)]
    agg = view.groupby('date')[['demand','forecast','stock','inbound']].sum()
    selected_sku=sorted(view.sku.unique())[0]
    item=view[view.sku.eq(selected_sku)].sort_values('date')
    fig,ax=plt.subplots(2,2,figsize=(14,8),layout='constrained')
    agg[['demand','forecast']].plot(ax=ax[0,0],title=f'{plant} | actual vs forecast')
    ax[0,0].plot(agg.index,agg.stock+agg.inbound,label='Available supply',alpha=.7);ax[0,0].legend()
    ax[0,1].plot(item.date,item.demand,label='Actual');ax[0,1].plot(item.date,item.forecast,label='Forecast')
    ax[0,1].fill_between(item.date,item.lower,item.upper,alpha=.2,label='90% target interval')
    ax[0,1].set_title(selected_sku+' | forecast interval');ax[0,1].legend()
    rows=shipments[shipments.plant.eq(plant)&shipments.downtime_days.gt(7)]
    if len(rows): rows.cause.value_counts().plot.barh(ax=ax[1,0],title='Downtime breaches by cause')
    view.groupby('date')[['revenue','pat']].sum().plot(ax=ax[1,1],title='Illustrative revenue and PAT (USD)')
    plt.show()
    display(queue[queue.plant.isin([plant,'Network'])].head(12))
def observe():
    events=pd.DataFrame(trace)
    display(events[['timestamp','alert_id','status','confidence','sources','latency_ms','generation_ms']].tail(15))
    events.groupby('status').size().plot.bar(title='Assistant outcomes',rot=20);plt.show()
    display(pd.DataFrame(trace[-1]['reranked']))
dashboard()
observe()

In [ ]:
import ipywidgets as w
from IPython.display import clear_output
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass
plant_picker=w.Dropdown(options=plants.plant.tolist(),description='Plant')
alert_picker=w.Dropdown(options=[('General question',None)]+[(f'{r.alert_id} | {r.plant} | {r.kind}',r.alert_id) for r in queue.itertuples()],value=case,description='Alert')
question=w.Text(value='What should the planner do?',description='Question')
refresh=w.Button(description='Refresh dashboard')
ask=w.Button(description='Run assistant',button_style='primary')
view_output=w.Output()
answer_output=w.Output()
def show_dashboard(_):
    with view_output:
        clear_output(wait=True)
        dashboard(plant_picker.value)
def show_answer(_):
    with answer_output:
        clear_output(wait=True)
        result,event=answer(question.value,alert_picker.value)
        print(result)
        observe()
refresh.on_click(show_dashboard)
ask.on_click(show_answer)
display(w.VBox([w.HBox([plant_picker,refresh]),view_output,alert_picker,question,ask,answer_output]))


## 6. MLOps handoff
The export includes model and dataset fingerprints, split dates, metric comparisons, policy snapshots and event traces. Compare forecast MAE against seasonal naive and monitor interval coverage on delayed actuals. Review distribution shifts before retraining, promote only after a chronological holdout and preserve the prior model for rollback. This notebook demonstrates a local registry manifest and a promotion check. Scheduling, approvals and notification delivery require a deployed service.

In [ ]:
import sklearn
from importlib.metadata import version, PackageNotFoundError
artifact_dir=Path('supply_risk_run')
artifact_dir.mkdir(exist_ok=True)
def package_version(name):
    try: return version(name)
    except PackageNotFoundError: return None
def export_run():
    tables={'plants':plants,'materials':materials,'suppliers':suppliers,'daily_facts':facts,
            'contracts':contracts,'shipments':shipments,'model_comparison':comparison,
            'forecast_backtest':test,'forecast_next_day':future,'alert_queue':queue}
    for name,table in tables.items(): table.to_csv(artifact_dir/f'{name}.csv',index=False)
    (artifact_dir/'knowledge.json').write_text(json.dumps(knowledge,indent=2))
    (artifact_dir/'retrieval_traces.json').write_text(json.dumps(trace,indent=2))
    model_file=artifact_dir/'champion.joblib'
    joblib.dump(pipelines.get(champion,{'baseline':'lag7'}),model_file)
    manifest={'model':champion,'model_sha256':hashlib.sha256(model_file.read_bytes()).hexdigest(),
              'dataset_sha256':hashlib.sha256((artifact_dir/'daily_facts.csv').read_bytes()).hexdigest(),
              'source':'synthetic', 'seed':SEED,'feature_schema':features,
              'train_end':str(train.date.max().date()),'validation_end':str(val.date.max().date()),
              'calibration_end':str(cal.date.max().date()),'test_end':str(test.date.max().date()),
              'test_mae':float(champion_mae),'seasonal_naive_test_mae':float(baseline_mae),
              'test_coverage':coverage,'nominal_coverage':.90,'interval_radius':radius,
              'promotion_check':bool(champion_mae<=baseline_mae and coverage>=.85),
              'versions':{p:package_version(p) for p in ['pandas','numpy','scikit-learn','xgboost','transformers']}}
    (artifact_dir/'manifest.json').write_text(json.dumps(manifest,indent=2))
    archive=Path('Supply_Risk_Colab_Run.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for f in sorted(artifact_dir.iterdir()): z.write(f,f.name)
    return archive
archive=export_run()
display(Markdown(f'[Download run outputs]({archive})'))
# In Colab use the Files panel or uncomment:
# from google.colab import files
# files.download(str(archive))
